# DSO607 — Homework 5
**Woojin Chae**

## Part (a): Numerical check of the counterexample

We check the matrices used in the written solution:

$$
\Omega=\begin{pmatrix}
1&-2/5&0\\-2/5&1&-2/5\\0&-2/5&1
\end{pmatrix},\qquad
\Sigma=\frac1{17}\begin{pmatrix}
21&10&4\\10&25&10\\4&10&21
\end{pmatrix}.
$$

The computations report both sets of eigenvalues, verify the inverse relationship, and check that $\omega_{13}=0$ while $\sigma_{13}=4/17\ne0$. These numerical checks supplement the analytical proof in the PDF.

The calculation is deterministic: no sampling or random seed is needed. Only NumPy is required. Run the notebook from top to bottom.

In [1]:
import numpy as np

np.set_printoptions(precision=10, suppress=True)
ATOL = 1e-12

# Define both matrices independently, using the analytical formulas.
Omega = np.array([
    [1.0, -2/5, 0.0],
    [-2/5, 1.0, -2/5],
    [0.0, -2/5, 1.0],
])
Sigma = np.array([
    [21.0, 10.0, 4.0],
    [10.0, 25.0, 10.0],
    [4.0, 10.0, 21.0],
]) / 17

print("Precision matrix Omega:\n", Omega)
print("\nCovariance matrix Sigma:\n", Sigma)

Precision matrix Omega:
 [[ 1.  -0.4  0. ]
 [-0.4  1.  -0.4]
 [ 0.  -0.4  1. ]]

Covariance matrix Sigma:
 [[1.2352941176 0.5882352941 0.2352941176]
 [0.5882352941 1.4705882353 0.5882352941]
 [0.2352941176 0.5882352941 1.2352941176]]


### Symmetry, positive definiteness, and inverses

For a real symmetric matrix, strictly positive eigenvalues imply positive definiteness. We use `eigvalsh`, which is designed for symmetric matrices. We also compare the two independently specified matrices with their numerical inverses and check both matrix products against $I_3$. Comparisons allow a small absolute tolerance for floating-point rounding.

In [2]:
np.testing.assert_allclose(Omega, Omega.T, rtol=0, atol=ATOL)
np.testing.assert_allclose(Sigma, Sigma.T, rtol=0, atol=ATOL)

eig_Omega = np.linalg.eigvalsh(Omega)
eig_Sigma = np.linalg.eigvalsh(Sigma)

print("Eigenvalues of Omega:", eig_Omega)
print("Eigenvalues of Sigma:", eig_Sigma)
print("Omega is positive definite:", bool(np.all(eig_Omega > 0)))
print("Sigma is positive definite:", bool(np.all(eig_Sigma > 0)))
assert np.all(eig_Omega > 0)
assert np.all(eig_Sigma > 0)

# Compare with the analytical eigenvalues from the written solution.
expected_eig_Omega = np.array([1 - 2*np.sqrt(2)/5, 1, 1 + 2*np.sqrt(2)/5])
np.testing.assert_allclose(eig_Omega, expected_eig_Omega, rtol=0, atol=ATOL)
np.testing.assert_allclose(eig_Sigma, np.sort(1/expected_eig_Omega), rtol=0, atol=ATOL)

I = np.eye(3)
np.testing.assert_allclose(np.linalg.inv(Omega), Sigma, rtol=0, atol=ATOL)
np.testing.assert_allclose(np.linalg.inv(Sigma), Omega, rtol=0, atol=ATOL)
np.testing.assert_allclose(Omega @ Sigma, I, rtol=0, atol=ATOL)
np.testing.assert_allclose(Sigma @ Omega, I, rtol=0, atol=ATOL)

print("\nOmega @ Sigma:\n", Omega @ Sigma)
print("Max absolute error in Omega @ Sigma - I:", np.max(np.abs(Omega @ Sigma - I)))
print("Max absolute error in Sigma @ Omega - I:", np.max(np.abs(Sigma @ Omega - I)))
print("All symmetry, eigenvalue, and inverse checks passed.")

Eigenvalues of Omega: [0.4343145751 1.           1.5656854249]
Eigenvalues of Sigma: [0.6386979045 1.           2.3024785661]
Omega is positive definite: True
Sigma is positive definite: True

Omega @ Sigma:
 [[ 1. -0. -0.]
 [-0.  1. -0.]
 [-0.  0.  1.]]
Max absolute error in Omega @ Sigma - I: 5.4858078863831267e-17
Max absolute error in Sigma @ Omega - I: 5.4858078863831267e-17
All symmetry, eigenvalue, and inverse checks passed.


### Marginal versus conditional dependence

Python uses zero-based indexing, so mathematical entry $(1,3)$ is accessed as `[0, 2]`.

For the Gaussian vector, the marginal and partial correlations are

$$\operatorname{Corr}(X_1,X_3)=\frac{\sigma_{13}}{\sqrt{\sigma_{11}\sigma_{33}}},\qquad
\operatorname{Corr}(X_1,X_3\mid X_2)=-\frac{\omega_{13}}{\sqrt{\omega_{11}\omega_{33}}}.$$

As an additional check, let $A=(1,3)$ and $B=(2)$. The conditional covariance is

$$\operatorname{Cov}(X_A\mid X_B)=\Sigma_{AA}-\Sigma_{AB}\Sigma_{BB}^{-1}\Sigma_{BA}.$$

We evaluate this formula by solving a linear system.

In [3]:
omega_13 = Omega[0, 2]
sigma_13 = Sigma[0, 2]
marginal_corr = sigma_13 / np.sqrt(Sigma[0, 0] * Sigma[2, 2])
partial_corr = -omega_13 / np.sqrt(Omega[0, 0] * Omega[2, 2])

A, B = [0, 2], [1]
Sigma_AA = Sigma[np.ix_(A, A)]
Sigma_AB = Sigma[np.ix_(A, B)]
Sigma_BB = Sigma[np.ix_(B, B)]
conditional_cov = Sigma_AA - Sigma_AB @ np.linalg.solve(Sigma_BB, Sigma_AB.T)

print(f"omega_13 = {omega_13:.10f}")
print(f"sigma_13 = {sigma_13:.10f} (4/17)")
print(f"Marginal correlation of X1 and X3 = {marginal_corr:.10f}")
print(f"Partial correlation given X2 = {partial_corr:.10f}")
print("\nConditional covariance of (X1, X3) given X2:\n", conditional_cov)

np.testing.assert_allclose(omega_13, 0, rtol=0, atol=ATOL)
np.testing.assert_allclose(sigma_13, 4/17, rtol=0, atol=ATOL)
assert not np.isclose(sigma_13, 0, rtol=0, atol=ATOL)
np.testing.assert_allclose(marginal_corr, 4/21, rtol=0, atol=ATOL)
np.testing.assert_allclose(partial_corr, 0, rtol=0, atol=ATOL)
np.testing.assert_allclose(conditional_cov, np.eye(2), rtol=0, atol=ATOL)
print("\nAll Part (a) numerical checks passed.")

omega_13 = 0.0000000000
sigma_13 = 0.2352941176 (4/17)
Marginal correlation of X1 and X3 = 0.1904761905
Partial correlation given X2 = -0.0000000000

Conditional covariance of (X1, X3) given X2:
 [[1. 0.]
 [0. 1.]]

All Part (a) numerical checks passed.


### Interpretation

Both matrices have strictly positive eigenvalues, and their inverse relationship holds to floating-point precision. The precision entry $\omega_{13}=0$, whereas $\sigma_{13}=4/17\approx0.2352941176$ and the marginal correlation is $4/21\approx0.1904761905$.

The conditional covariance of $(X_1,X_3)$ given $X_2$ is $I_2$, and the partial correlation is zero. Because the vector is jointly Gaussian, this means $X_1$ and $X_3$ are conditionally independent given $X_2$. They are marginally dependent because their covariance is nonzero. Thus, a zero off-diagonal precision entry does **not** imply a zero covariance entry.

### AI-use disclosure

I used OpenAI ChatGPT/Codex to help develop and check the Python implementation and explanatory text for Part (a).